# GIAI ĐOẠN 2: TIỀN HUẤN LUYỆN BASE MODEL (PRE-TRAINING) & PROFILING KV CACHE

> **Dự án:** Xây dựng Large Language Model from Scratch (Clean Architecture)  
> **Môi trường thực thi mục tiêu:** **Kaggle GPU (Tesla T4 16GB VRAM)**  
> **Đặc tả theo `TASK_new.md`:**  
> - **Tokenizer:** Sử dụng Byte-Fallback BPE Tokenizer chuẩn 8.192 tokens (8K) được nạp từ `checkpoints/Tokenizer/` (hoặc tạo từ `LLM_00_train_tokenizer_and_prepare_data.ipynb`). Tuyệt đối không dùng lại checkpoint legacy Vocab 261.  
> - **Weights & Biases (W&B):** Tự động theo dõi các metrics: `train/loss`, `val/loss`, `val/perplexity`, `train/learning_rate`, `system/peak_vram_mb` và bảng text samples; cấu hình rõ ràng Project Name & Run Name chuyên biệt cho Kaggle.  
> - **Cơ chế Auto-Resume trên Kaggle:** Tự động phát hiện checkpoint gần nhất (`last_checkpoint.pt` hoặc `step_*.pt`) để tiếp tục huấn luyện nếu session bị ngắt quãng, giữ đường dẫn tuyệt đối `/kaggle/working/Prj_LLM/checkpoints/` để chống lệch CWD.  
> - **Kỹ thuật tối ưu VRAM:** Mixed Precision (`torch.amp.autocast`), Gradient Accumulation (effective batch 128 sequences), Gradient Clipping (1.0).  
> - **Đo đạc định lượng (Profiling) KV Cache:** Tốc độ (Tokens/s, TTFT, TPOT) và Bộ nhớ (Peak VRAM) khi **Bật KV Cache vs Tắt KV Cache** tại 4 mốc ngữ cảnh $L \in \{128, 512, 1024, 2048\}$ kèm bảng tổng hợp và biểu đồ trực quan.


## 1. Thiết lập Môi trường Kaggle & Kết nối Mã nguồn

In [ ]:
# ==============================================================================
# 1. THIẾT LẬP MÔI TRƯỜNG KAGGLE CLOUD & CLONE REPO CHUẨN XÁC
# ==============================================================================
import os
import sys

# Nhận diện môi trường Kaggle và điều hướng an toàn
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    if not os.path.exists('/kaggle/working/Prj_LLM'):
        %cd /kaggle/working
        !git clone -b dev https://github.com/TyTrinh/Prj_LLM.git
        %cd /kaggle/working/Prj_LLM
        !pip install -q -e .
    else:
        %cd /kaggle/working/Prj_LLM
        !git pull origin dev
    REPO_ROOT = '/kaggle/working/Prj_LLM'
else:
    REPO_ROOT = os.path.abspath('.')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f'📂 Thư mục làm việc hiện tại: {os.getcwd()}')
!pip install -q wandb datasets matplotlib pandas
!nvidia-smi


## 2. Kết nối Weights & Biases (W&B) với Tên Project Chuyên biệt cho Kaggle

In [ ]:
# ==============================================================================
# 2. KET NOI VOI WEIGHTS & BIASES (W&B)
# ==============================================================================
import os
import wandb

WANDB_PROJECT = "LLM_Stage2_Pretrain_BaseModel_Kaggle"
WANDB_RUN_NAME = "Pretrain-BaseModel-BPE8K-Kaggle-T4"

USE_WANDB = True
WANDB_MODE = "online"  # "online", "offline", hoặc "disabled"
WANDB_API_KEY = None

# 1. Quet tim API Key tu Kaggle Secrets (kiem tra cac ten Label pho bien)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_name in ["WANDB_API_KEY", "wandb_api_key", "wandb_key", "wandb", "WANDB"]:
        try:
            val = user_secrets.get_secret(secret_name)
            if val and val.strip():
                WANDB_API_KEY = val.strip()
                print(f"🔑 Da tim thay secret: '{secret_name}' tu Kaggle Secrets!")
                break
        except Exception:
            continue
except Exception:
    pass

# 2. Du phong doc tu bien moi truong he thong
if not WANDB_API_KEY:
    for env_name in ["WANDB_API_KEY", "wandb_api_key", "WANDB_KEY"]:
        if os.environ.get(env_name):
            WANDB_API_KEY = os.environ.get(env_name).strip()
            print(f"🔑 Da tim thay key tu bien moi truong: {env_name}")
            break

# 3. Tien hanh Login tu dong (tranh bi chan boi prompt nhap tay)
if USE_WANDB and WANDB_MODE == "online":
    if WANDB_API_KEY:
        try:
            os.environ["WANDB_API_KEY"] = WANDB_API_KEY
            wandb.login(key=WANDB_API_KEY, relogin=True)
            print(f"✅ W&B Login thanh cong tu dong! Du an: {WANDB_PROJECT} | Run: {WANDB_RUN_NAME}")
        except Exception as e:
            print(f"⚠️ Khong the login W&B voi key: {e}. Chuyen sang offline.")
            WANDB_MODE = "offline"
            os.environ["WANDB_MODE"] = "offline"
    else:
        print("⚠️ CANH BAO: Chua tim thay Secret Key duoc gan vao Notebook!")
        print("   👉 NGUYEN NHAN PHO BIEN:")
        print("      1. Ban da tao Secret trong Kaggle nhung CHUA TICH CHON checkbox o popup 'Add-ons -> Secrets' de kich hoat cho notebook nay.")
        print("      2. Ten Label cua Secret khac voi 'WANDB_API_KEY' (vi du: 'wandb', 'wandb_key').")
        print("   ℹ️ Dang tu dong chuyen sang WANDB_MODE='offline' de cell chay thong suot khong bi dung hoi nhap phim!")
        WANDB_MODE = "offline"
        os.environ["WANDB_MODE"] = "offline"


## 3. Nạp Tokenizer BPE 8K & Dữ liệu Huấn luyện Đã Chuẩn hóa

In [ ]:
import numpy as np
import json
from src.tokenizer import BytePairTokenizer

# Nạp Tokenizer đã huấn luyện đầy đủ 8.192 tokens
tokenizer = BytePairTokenizer()

# Ưu tiên nạp từ checkpoints/Tokenizer/ (artifact của notebook LLM_00)
vocab_candidates = [
    'checkpoints/Tokenizer/vocab.json',
    'checkpoints/tokenizer_8k/vocab.json',
    'checkpoints/vocab.json',
]
merges_candidates = [
    'checkpoints/Tokenizer/merges.txt',
    'checkpoints/tokenizer_8k/merges.txt',
    'checkpoints/merges.txt',
]

vocab_file = next((f for f in vocab_candidates if os.path.exists(f)), None)
merges_file = next((f for f in merges_candidates if os.path.exists(f)), None)

if vocab_file and merges_file:
    print(f'Đang nạp từ điển Tokenizer từ: {vocab_file} và {merges_file}...')
    tokenizer.load(vocab_file, merges_file)
else:
    print('⚠️ Chưa tìm thấy file vocab.json/merges.txt! Sử dụng từ điển cơ bản...')

ACTUAL_VOCAB_SIZE = len(tokenizer.vocab)
print(f'✅ Kích thước từ điển thực tế (Vocab Size): {ACTUAL_VOCAB_SIZE:,}')

# Kiểm tra khớp đúng đặc tả Vocab 8.192 theo TASK_new.md
if ACTUAL_VOCAB_SIZE == 8192:
    print('🎯 XÁC NHẬN: Tokenizer khớp 100% chuẩn Vocab 8.192 (8K)')
else:
    print(f'Lưu ý: Vocab hiện tại là {ACTUAL_VOCAB_SIZE} (khác 8.192). ModelConfig sẽ tự động khớp theo kích thước này.')

# Đường dẫn dữ liệu binary memmap (tự động quét Kaggle Datasets trong /kaggle/input/ hoặc repo local)
data_search_candidates = [
    ('checkpoints/Tokenizer/train.bin', 'checkpoints/Tokenizer/val.bin'),
    ('data/train.bin', 'data/val.bin'),
]

if os.path.exists('/kaggle/input'):
    import glob
    found_train = glob.glob('/kaggle/input/**/train.bin', recursive=True)
    for t_path in found_train:
        v_path = t_path.replace('train.bin', 'val.bin')
        data_search_candidates.insert(0, (t_path, v_path))

train_bin_path = None
val_bin_path = None
for t_path, v_path in data_search_candidates:
    if os.path.exists(t_path):
        train_bin_path = t_path
        val_bin_path = v_path if os.path.exists(v_path) else None
        break

if not train_bin_path:
    raise FileNotFoundError(
        '❌ Không tìm thấy file dữ liệu train.bin!\n'
        '👉 HÃY CHỌN 1 TRONG 2 CÁCH:\n'
        '   1. Upload dataset lên Kaggle rồi bấm "+ Add Input" ở góc phải.\n'
        '   2. Hoặc mở notebook LLM_00_train_tokenizer_and_prepare_data.ipynb trên Kaggle và bấm Run All một lần.'
    )
else:
    print(f'📂 Đang nạp dữ liệu từ: {train_bin_path}')
    train_data = np.memmap(train_bin_path, dtype=np.uint16, mode='r')
    val_data = np.memmap(val_bin_path, dtype=np.uint16, mode='r') if val_bin_path and os.path.exists(val_bin_path) else None
    print('✅ Đã nạp thành công dữ liệu nhị phân:')
    print(f'  • Train tokens: {len(train_data):,} tokens ({os.path.getsize(train_bin_path) / (1024**2):.1f} MB)')
    if val_data is not None:
        print(f'  • Val tokens:   {len(val_data):,} tokens ({os.path.getsize(val_bin_path) / (1024**2):.1f} MB)')


## 4. Khởi tạo Kiến trúc Transformer Lõi (Fit 16GB VRAM, RoPE Hỗ trợ $L=2048$)

In [ ]:
import torch
from src.config import ModelConfig
from src.model import GPTModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# Cấu hình mô hình tối ưu theo TASK_new.md
# - block_size=2048: Cấp dải tần RoPE và Causal Mask rộng đến 2048 để phục vụ đo Profiling
# - TRAIN_BLOCK_SIZE=512: Độ dài ngữ cảnh huấn luyện thực tế để bao trọn 100% câu chuyện TinyStories mà tiết kiệm VRAM
model_cfg = ModelConfig(
    vocab_size=ACTUAL_VOCAB_SIZE,
    block_size=2048,
    n_layers=8,
    n_heads=8,
    d_model=512,
    tie_weights=True,
)

model = GPTModel(model_cfg).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"🚀 GPTModel đã khởi tạo thành công trên thiết bị: {device}")
print(f"• Tổng tham số: {total_params / 1e6:.2f}M")
print(f"• Vocab Size:   {model_cfg.vocab_size:,}")
print(f"• Weight Tying: {model_cfg.tie_weights} (Tiết kiệm ~{model_cfg.vocab_size * model_cfg.d_model / 1e6:.2f}M tham số LM Head)")


## 5. Vòng lặp Huấn luyện (AMP, Grad Accumulation, WSD Scheduler (Warmup-Stable-Decay), W&B, Auto-Resume)

> **Đặc tính Auto-Resume:** Sử dụng đường dẫn tuyệt đối an toàn trên Kaggle (`/kaggle/working/Prj_LLM/checkpoints/`). Nếu phiên GPU Kaggle bị timeout hoặc ngắt kết nối, chỉ cần chạy lại cell này, mô hình sẽ tự động nạp `last_checkpoint.pt` hoặc snapshot step gần nhất để tiếp tục mà không train lại từ đầu.


In [ ]:
import os
import glob
import math
import time
import torch.nn as nn
from src.utils.checkpoint import CheckpointManager, CheckpointRollbackManager
from src.utils.tracker import WandbTracker

# ==============================================================================
# BẢNG SIÊU THAM SỐ HUẤN LUYỆN CHUẨN CÔNG NGHIỆP (T4 x 2 DDP / 1.5 EPOCHS)
# ==============================================================================
TRAIN_BLOCK_SIZE = 512         # Chiều dài sequence huấn luyện
PER_GPU_BATCH_SIZE = 4         # Microbatch size trên mỗi GPU (hoặc 1 GPU x 2)
BATCH_SIZE = 8                 # Local microbatch (nếu chạy 1 GPU: 8, nếu 2 GPU: 4x2)
GRAD_ACCUM_STEPS = 8           # Tích lũy gradient -> Effective batch = 64 sequences
TOKENS_PER_UPDATE = 64 * 512   # 32,768 tokens / optimizer update

# Lịch trình tốc độ học: WSD Scheduler (Warmup - Stable - Decay)
MAX_STEPS = 4600               # 4.600 steps (~150.7M tokens = 1.5 Epochs)
PEAK_LR = 3e-4                 # Tốc độ học cực đại (eta_max)
MIN_LR = 3e-5                  # Tốc độ học cực tiểu (0.1 * eta_max)
WARMUP_RATIO = 0.03            # 1% - 5%: Warmup tuyến tính (3% = 138 steps)
DECAY_RATIO = 0.10             # 10% cuối cùng: Decay nhanh (10% = 460 steps)
# -> Stable Phase: Chiếm ~87% thời gian (duy trì PEAK_LR tuyệt đối)

# Tần suất đánh giá và lưu checkpoint
EVAL_INTERVAL = 500            # Đánh giá validation mỗi 500 steps
SAVE_CHECKPOINT_EVERY = 500   # Lưu checkpoint định kỳ mỗi 500 steps
GRAD_CLIP = 1.0                # Gradient clipping chống bùng nổ đạo hàm

# Thư mục lưu checkpoint tuyệt đối an toàn
CHECKPOINT_DIR = os.path.abspath('checkpoints')
LAST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt')
BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, 'base_model.pt')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Khởi tạo Optimizer AdamW chuẩn: Betas=(0.9, 0.95), Weight decay=0.1
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=PEAK_LR,
    weight_decay=0.1,
    betas=(0.9, 0.95)
)
use_amp = (device == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

# ==============================================================================
# CƠ CHẾ AUTO-RESUME THÔNG MINH (KHI BỊ DISCONNECT HOẶC STOP DỞ DANG)
# ==============================================================================
# Khởi tạo CheckpointRollbackManager giám sát ứng cứu tự động (Rollback Strategy)
rollback_mgr = CheckpointRollbackManager(
    checkpoint_dir=CHECKPOINT_DIR,
    threshold_factor=3.0,
    lr_reduction_ratio=0.20,
    recovery_steps=50,
)
if candidate_ckpt:
    rollback_mgr.register_safe_checkpoint(candidate_ckpt)

start_step = 0
best_val_loss = float('inf')

candidate_ckpt = None
if os.path.exists(LAST_CKPT_PATH):
    candidate_ckpt = LAST_CKPT_PATH
else:
    all_step_ckpts = glob.glob(os.path.join(CHECKPOINT_DIR, 'step_*.pt'))
    if all_step_ckpts:
        candidate_ckpt = max(all_step_ckpts, key=os.path.getmtime)

if candidate_ckpt and os.path.exists(candidate_ckpt):
    print(f'🔄 [PHÁT HIỆN CHECKPOINT DỞ DANG] Đang khôi phục từ: {candidate_ckpt}...')
    meta = CheckpointManager.load_checkpoint(
        candidate_ckpt,
        model=model,
        optimizer=optimizer,
        scaler=scaler if use_amp else None,
        device=device,
    )
    start_step = meta.get('step', 0)
    best_val_loss = meta.get('best_val_loss', float('inf'))
    print(f'✅ Khôi phục thành công! Sẽ chạy tiếp tục từ Step {start_step + 1:,} đến Step {MAX_STEPS:,}')
    print(f'• Best Val Loss trước đó: {best_val_loss:.4f}')
else:
    print(f'🌱 Bắt đầu huấn luyện mới hoàn toàn từ Step 0 đến Step {MAX_STEPS:,} (1.5 Epochs).')

# Khởi tạo Tracker W&B chuyên biệt cho Giai đoạn Pretraining
tracker = WandbTracker(
    project_name='LLM_Stage2_Pretrain_BaseModel',
    run_name=f'Pretrain-BPE8K-4600Steps-Step{start_step}',
    config={
        'model_parameters': '~30M',
        'vocab_size': model_cfg.vocab_size,
        'd_model': model_cfg.d_model,
        'n_layers': model_cfg.n_layers,
        'n_heads': model_cfg.n_heads,
        'train_block_size': TRAIN_BLOCK_SIZE,
        'effective_batch': 64,
        'tokens_per_update': TOKENS_PER_UPDATE,
        'peak_lr': PEAK_LR,
        'min_lr': MIN_LR,
        'warmup_ratio': WARMUP_RATIO,
        'decay_ratio': DECAY_RATIO,
        'scheduler': 'WSD (Warmup-Stable-Decay)',
        'max_steps': MAX_STEPS,
        'target_epochs': 1.5,
    },
    enabled=USE_WANDB,
    mode=WANDB_MODE,
)

# Nạp dữ liệu qua Memmap (ưu tiên thư mục checkpoints/Tokenizer nếu có)
train_data_path = 'checkpoints/Tokenizer/train.bin' if os.path.exists('checkpoints/Tokenizer/train.bin') else 'data/train.bin'
val_data_path = 'checkpoints/Tokenizer/val.bin' if os.path.exists('checkpoints/Tokenizer/val.bin') else 'data/val.bin'
train_data = np.memmap(train_data_path, dtype=np.uint16, mode='r')
val_data = np.memmap(val_data_path, dtype=np.uint16, mode='r')

print(f'• Nạp Train Binary: {train_data_path} ({len(train_data):,} tokens)')
print(f'• Nạp Val Binary:   {val_data_path} ({len(val_data):,} tokens)')

def get_batch(split='train'):
    data = train_data if split == 'train' else val_data
    ix = np.random.randint(0, len(data) - TRAIN_BLOCK_SIZE, size=BATCH_SIZE)
    x = torch.stack([torch.from_numpy(data[i : i + TRAIN_BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + TRAIN_BLOCK_SIZE].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

from src.utils.scheduler import get_wsd_lr

def get_lr(it):
    """
    Lịch trình WSD (Warmup - Stable - Decay):
    1. Warmup (0 -> 3%): Tăng tuyến tính 0 -> PEAK_LR (138 steps)
    2. Stable (3% -> 90%): Giữ cố định 100% PEAK_LR (hấp thụ tri thức tối đa, không decay sớm)
    3. Decay (90% -> 100%): Annealing dốc theo Cosine về MIN_LR (460 steps cuối)
    """
    return get_wsd_lr(
        step=it,
        max_steps=MAX_STEPS,
        max_lr=PEAK_LR,
        min_lr=MIN_LR,
        warmup_ratio=WARMUP_RATIO,
        decay_ratio=DECAY_RATIO,
        decay_type="cosine"
    )

@torch.no_grad()
def estimate_loss(eval_iters=50):
    model.eval()
    losses = {}
    criterion = nn.CrossEntropyLoss()
    for split in ['train', 'val']:
        loss_sum = 0.0
        for _ in range(eval_iters):
            x, y = get_batch(split)
            with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss_sum += loss.item()
        losses[split] = loss_sum / eval_iters
    model.train()
    return losses


In [ ]:
# ==============================================================================
# VÒNG LẶP HUẤN LUYỆN CHÍNH (TRAINING LOOP - 4.600 STEPS / 1.5 EPOCHS)
# ==============================================================================
criterion = nn.CrossEntropyLoss()
model.train()
t0 = time.time()

for step in range(start_step, MAX_STEPS):
    base_lr = get_lr(step)
    effective_lr = base_lr * rollback_mgr.current_lr_multiplier
    for param_group in optimizer.param_groups:
        param_group['lr'] = effective_lr

    optimizer.zero_grad()
    accum_loss = 0.0

    # Tích lũy gradient qua GRAD_ACCUM_STEPS
    for micro_step in range(GRAD_ACCUM_STEPS):
        x, y = get_batch('train')
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            logits = model(x)
            micro_loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            micro_loss_scaled = micro_loss / GRAD_ACCUM_STEPS

        accum_loss += micro_loss.item()
        if use_amp:
            scaler.scale(micro_loss_scaled).backward()
        else:
            micro_loss_scaled.backward()

    current_train_loss = accum_loss / GRAD_ACCUM_STEPS

    # 1. Gradient Clipping & Đo lường Norm để phát hiện sự cố (Anomaly Detection)
    if use_amp:
        scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRAD_CLIP * rollback_mgr.current_clip_multiplier
    ).item()

    # 2. Phát hiện ngưỡng bất thường: ||g|| > Threshold * EMA hoặc NaN/Inf
    if rollback_mgr.check_gradient_anomaly(grad_norm) or math.isnan(current_train_loss):
        print(f"⚠️ [SỰ CỐ GRADIENT] Phát hiện bất thường tại Step {step+1}: GradNorm={grad_norm:.2f}, Loss={current_train_loss:.4f}")
        rollback_mgr.trigger_rollback(
            model=model,
            optimizer=optimizer,
            scaler=scaler if use_amp else None,
            device=device,
            failed_step=step+1,
            reason=f"Gradient norm anomaly spike ({grad_norm:.2f})"
        )
        optimizer.zero_grad()
        continue

    if use_amp:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()

    # Theo dõi chu kỳ hồi phục siêu tham số
    rollback_mgr.step_recovery(optimizer)

    # Log tiến trình mỗi 50 steps
    if (step + 1) % 50 == 0 or step == start_step:
        dt = time.time() - t0
        t0 = time.time()
        current_vram = torch.cuda.max_memory_allocated() / (1024**2) if use_amp else 0.0
        current_epoch_progress = ((step + 1) * TOKENS_PER_UPDATE) / len(train_data)
        
        print(f'Step {step + 1:5d}/{MAX_STEPS} ({current_epoch_progress:.2f} Epochs) | Loss: {current_train_loss:.4f} | LR: {lr:.6f} | VRAM: {current_vram:.1f}MB | {dt:.2f}s/50steps')
        
        tracker.log({
            'train/loss': current_train_loss,
            'train/learning_rate': lr,
            'train/epoch_progress': current_epoch_progress,
            'system/peak_vram_mb': current_vram,
        }, step=step + 1)

    # ==========================================================================
    # ĐÁNH GIÁ VALIDATION & LƯU CHECKPOINTS ĐỊNH KỲ (MỖI 500 STEPS)
    # ==========================================================================
    if (step + 1) % SAVE_CHECKPOINT_EVERY == 0 or (step + 1) == MAX_STEPS:
        current_step = step + 1
        losses = estimate_loss()
        val_ppl = math.exp(min(losses['val'], 20))
        print(f'\n>>> [ĐÁNH GIÁ STEP {current_step}] Train Loss = {losses["train"]:.4f}, Val Loss = {losses["val"]:.4f}, Val Perplexity = {val_ppl:.2f}')

        tracker.log({
            'val/loss': losses['val'],
            'val/perplexity': val_ppl,
        }, step=current_step)

        # 1. Cập nhật best_val_loss và lưu base_model.pt
        is_new_best = losses['val'] < best_val_loss
        if is_new_best:
            best_val_loss = losses['val']
            CheckpointManager.save_base_model(model, BEST_CKPT_PATH)
            print(f'🌟 [KỶ LỤC MỚI] Đã lưu Base Model tốt nhất tại: {BEST_CKPT_PATH}')

        # 2. Luôn lưu last_checkpoint.pt để sẵn sàng Auto-Resume bất cứ lúc nào
        CheckpointManager.save_checkpoint(
            LAST_CKPT_PATH,
            model=model,
            optimizer=optimizer,
            scaler=scaler if use_amp else None,
            step=current_step,
            best_val_loss=best_val_loss,
            stage='pretrain',
        )
        print(f'💾 [Auto-Resume] Đã cập nhật: {LAST_CKPT_PATH}')

        # 3. Lưu checkpoint riêng theo từng mốc 500 steps (step_500.pt, step_1000.pt, ...)
        step_ckpt_path = os.path.join(CHECKPOINT_DIR, f'step_{current_step}.pt')
        CheckpointManager.save_checkpoint(
            step_ckpt_path,
            model=model,
            optimizer=optimizer,
            scaler=scaler if use_amp else None,
            step=current_step,
            best_val_loss=best_val_loss,
            stage='pretrain',
        )
        print(f'📦 [Periodic Snapshot] Đã lưu mốc Step {current_step} tại: {step_ckpt_path}')
        print('-' * 75)


## 6. Đo đạc Định lượng (Profiling): Bật KV Cache vs. Tắt KV Cache ($L \in \{128, 512, 1024, 2048\}$)

In [ ]:
from src.utils.profiler import KVCacheProfiler
import pandas as pd
import json

# Nạp trọng số mô hình tốt nhất vừa huấn luyện
if os.path.exists(BEST_CKPT_PATH):
    CheckpointManager.load_base_model(model, BEST_CKPT_PATH, device=device)
model.eval()

context_lengths = [128, 512, 1024, 2048]
benchmark_results = []

print("=" * 80)
print("📊 TIẾN HÀNH ĐO ĐẠC ĐỊNH LƯỢNG: CÓ KV CACHE VS. KHÔNG CÓ KV CACHE")
print("=" * 80)

for L in context_lengths:
    print(f"\n>>> Đang đo mốc Context Length L = {L} (Sinh 64 tokens)...")
    
    # 1. Đo khi BẬT KV Cache
    res_on = KVCacheProfiler.measure(model, prompt_length=L, generate_tokens=64, use_kv_cache=True, device=device)
    print(f"  [BẬT KV CACHE]  -> Tốc độ: {res_on.tokens_per_sec:6.2f} tok/s | TTFT: {res_on.ttft_ms:6.2f} ms | TPOT: {res_on.tpot_ms:6.2f} ms | VRAM: {res_on.peak_vram_mb:6.2f} MB")
    benchmark_results.append(vars(res_on))

    # 2. Đo khi TẮT KV Cache
    res_off = KVCacheProfiler.measure(model, prompt_length=L, generate_tokens=64, use_kv_cache=False, device=device)
    print(f"  [TẮT KV CACHE]  -> Tốc độ: {res_off.tokens_per_sec:6.2f} tok/s | TTFT: {res_off.ttft_ms:6.2f} ms | TPOT: {res_off.tpot_ms:6.2f} ms | VRAM: {res_off.peak_vram_mb:6.2f} MB")
    benchmark_results.append(vars(res_off))

# Lưu kết quả đo định lượng dạng JSON
figures_dir = os.path.join(REPO_ROOT, 'docs/figures')
os.makedirs(figures_dir, exist_ok=True)
json_output_path = os.path.join(figures_dir, 'kv_cache_metrics.json')
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_results, f, indent=4)

# Hiển thị bảng tổng hợp so sánh trực quan (Pandas DataFrame)
table_rows = []
for L in context_lengths:
    on = next(r for r in benchmark_results if r["context_length"] == L and r["use_kv_cache"])
    off = next(r for r in benchmark_results if r["context_length"] == L and not r["use_kv_cache"])
    speedup = on["tokens_per_sec"] / max(off["tokens_per_sec"], 1e-6)
    table_rows.append({
        "Context (L)": L,
        "KV Cache": "BẬT",
        "Speed (tok/s)": on["tokens_per_sec"],
        "TTFT (ms)": on["ttft_ms"],
        "TPOT (ms)": on["tpot_ms"],
        "Peak VRAM (MB)": on["peak_vram_mb"],
        "Speedup": f"{speedup:.2f}x",
    })
    table_rows.append({
        "Context (L)": L,
        "KV Cache": "TẮT",
        "Speed (tok/s)": off["tokens_per_sec"],
        "TTFT (ms)": off["ttft_ms"],
        "TPOT (ms)": off["tpot_ms"],
        "Peak VRAM (MB)": off["peak_vram_mb"],
        "Speedup": "1.00x",
    })

df_metrics = pd.DataFrame(table_rows)
display(df_metrics)

# Đẩy bảng metrics lên Weights & Biases
tracker.log_text_samples("benchmark/kv_cache_comparison", table_rows)


## 7. Xuất Biểu đồ So sánh Hiệu năng & Artifacts Checkpoint 2

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

metrics = [
    ("tokens_per_sec", "Tốc độ sinh (Tokens/s) - [Càng cao càng tốt]", "Tokens/giây"),
    ("ttft_ms", "First Token Latency - TTFT (ms) - [Càng thấp càng tốt]", "Mili-giây (ms)"),
    ("tpot_ms", "Next Token Latency - TPOT (ms) - [Càng thấp càng tốt]", "Mili-giây (ms)"),
    ("peak_vram_mb", "Chiếm dụng Peak VRAM (MB)", "Megabytes (MB)"),
]

for idx, (m_key, title, ylabel) in enumerate(metrics):
    ax = axes[idx]
    with_kv = [next(r[m_key] for r in benchmark_results if r["context_length"] == L and r["use_kv_cache"]) for L in context_lengths]
    without_kv = [next(r[m_key] for r in benchmark_results if r["context_length"] == L and not r["use_kv_cache"]) for L in context_lengths]

    x_indices = np.arange(len(context_lengths))
    width = 0.35

    ax.bar(x_indices - width/2, with_kv, width=width, label="Bật KV Cache", color="#2ca02c")
    ax.bar(x_indices + width/2, without_kv, width=width, label="Tắt KV Cache", color="#d62728")

    ax.set_xticks(x_indices)
    ax.set_xticklabels([f"L={L}" for L in context_lengths], fontweight="bold")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.7)

plt.suptitle("KẾT QUẢ ĐO ĐẠC THỰC NGHIỆM ĐỊNH LƯỢNG HIỆU NĂNG KV CACHE", fontsize=14, fontweight="bold")
plt.tight_layout()
plot_path = os.path.join(figures_dir, "kv_cache_benchmark.png")
plt.savefig(plot_path, dpi=300)
plt.show()
print(f"📊 Đã xuất và lưu biểu đồ tại: {plot_path}")

# Log biểu đồ lên W&B nếu đang bật
if tracker.enabled and tracker.run:
    wandb.log({"benchmark/chart": wandb.Image(plot_path)})


## 8. Demo Sinh Văn Bản Tự Hồi Quy & Đóng W&B Session

In [ ]:
prompts = [
    "Once upon a time, there was a little girl named Lily",
    "One sunny day, a little dog found a magic ball",
    "In a small village near the mountains",
]

print("=" * 75)
print("✨ DEMO SINH VĂN BẢN VỚI BASE MODEL (KV CACHE ENABLED)")
print("=" * 75)

demo_samples = []
for p in prompts:
    input_ids = torch.tensor([tokenizer.encode(p)], device=device)
    output_ids = model.generate(
        input_ids,
        max_new_tokens=120,     # Độ dài đủ để hoàn chỉnh câu chuyện
        temperature=0.8,
        top_k=40,
        top_p=0.9,
        use_kv_cache=True,
    )
    gen_text = tokenizer.decode(output_ids[0].tolist())
    print(f"\n[Prompt]: {p}")
    print(f"[Generated]:\n{gen_text}")
    print("-" * 50)
    demo_samples.append({"prompt": p, "generated_text": gen_text})

# Log các mẫu sinh văn bản lên W&B
tracker.log_text_samples("demo/generations", demo_samples)

# Kết thúc theo dõi W&B
tracker.finish()
print("🎉 Toàn bộ quá trình Pre-training & Profiling KV Cache đã hoàn tất thành công!")


## 9. Đóng gói Artifacts Checkpoint 2 Tải về Máy Local (1-Click Zip)

In [ ]:
# Tạo gói zip chứa base_model.pt, last_checkpoint.pt và biểu đồ benchmark để tải về
export_dir = '/kaggle/working/export_stage2' if IS_KAGGLE else './export_stage2'
zip_out = '/kaggle/working/giai_doan_2_results.zip' if IS_KAGGLE else './giai_doan_2_results.zip'

!mkdir -p {export_dir}
!cp {BEST_CKPT_PATH} {export_dir}/ 2>/dev/null || true
!cp {LAST_CKPT_PATH} {export_dir}/ 2>/dev/null || true
!cp -r {figures_dir}/* {export_dir}/ 2>/dev/null || true
!zip -r {zip_out} {export_dir}

print(f"✅ Đã tạo gói kết quả tại: {zip_out}")

# Hiển thị nút download trực tiếp
from IPython.display import FileLink
if IS_KAGGLE:
    %cd /kaggle/working
    display(FileLink('giai_doan_2_results.zip'))
